# AMD-Latent
### UMAP visualizer of latent space

---

The current notebook loads UMAP extractions and colors them differently to visualize different information all at once.

# Packages import and functions defining

Loading the required packages and definition of functions.

In [ ]:
# PACKAGES IMPORT

# System packages
import os
import numpy as np
import pandas as pd
from pathlib import Path

# Jupyter packages
from ipywidgets import widgets
from amdlatent.utils.widgets import toggleoptions, filechooser, slideroptions, checkboxoptions

# Dataset packages
from amdlatent.dataset.audit import naneval
from amdlatent.dataset.extraction import filteringimages

# Visualization packages
from amdlatent.utils.visualization import coloringextr, umapdisplay

# Statistical analysis packages
import pickle as pkl

# Path checker
if Path().resolve().name != "AMD-Latent": os.chdir(os.path.join(Path().resolve().parents[1]))

# Code execution

Select the following options for running the script and execute it.

In [ ]:
# OPTIONS SELECTION

dirselect = filechooser("Select the <b>AMD Study</b> folder:")
clipselect = toggleoptions(['RETCLIP', 'CLIP', 'ResNet50'], "Select the <b>model</b> to use:")

visualselect = checkboxoptions(['View', 'Drusen', 'Pigment', 'Geographic Atrophy', 'Neovascular', 'Glaucoma', 'Diabetes', 'AMD Score', 'Classifier', 'Ground Truth', 'Set Split', 'Feature'], "Select the <b>visualization options</b> to display:")

featurebox = widgets.Output()
display(featurebox)
featureselect = None

def _updatefeaturetoggle(change=None):
    global featureselect
    if _is_checked("Feature"):
        if featureselect is None:
            with featurebox:
                featurebox.clear_output(wait=True)
                featureselect = slideroptions(
                    "Select the <b>feature number</b> to evaluate:", 0, 1024, 1, 609
                )
    else:
        if featureselect is not None:
            featurebox.clear_output(wait=True)
            featureselect = None

for cb in visualselect:
    cb.observe(_updatefeaturetoggle, names="value")
_updatefeaturetoggle()

def _is_checked(option_name):
    return any(cb.description == option_name and cb.value for cb in visualselect)

classifierbox = widgets.Output()
display(classifierbox)
classifierselect = None

def _updateclassifiertoggle(change=None):
    global classifierselect
    if _is_checked("Classifier"):
        if classifierselect is None:
            with classifierbox:
                classifierbox.clear_output(wait=True)
                classifierselect = toggleoptions(['Random Forest', 'AdaBoost', 'SVM', 'SVM Linear', 'MLP'], "Select the <b>classifier</b> to use:")
    else:
        if classifierselect is not None:
            classifierbox.clear_output(wait=True)
            classifierselect = None

for cb in visualselect:
    cb.observe(_updateclassifiertoggle, names="value")
_updateclassifiertoggle()

charactbox = widgets.Output()
display(charactbox)
splitselect = None
classselect = None

def _updatecharacttoggle(change=None):
    global splitselect
    global classselect
    if _is_checked("Classifier") or _is_checked("Ground Truth") or _is_checked("Set Split") or _is_checked("Feature"):
        if splitselect is None and classselect is None:
            with charactbox:
                charactbox.clear_output(wait=True)
                splitselect = toggleoptions(["Couples-wise", "Patient-wise", "Couples stratified-wise"],"Select the <b>dataset split type</b>:")
                classselect = toggleoptions(['All', 'Binary No', 'Binary Late'], "Select the <b>classification type</b> to use:")
    else:
        if splitselect is not None and classselect is not None:
            charactbox.clear_output(wait=True)
            splitselect = None
            classselect = None

for cb in visualselect:
    cb.observe(_updatecharacttoggle, names="value")
_updatecharacttoggle()

In [ ]:
# SCRIPT EXECUTION

metadata = {
    "directory": dirselect.selected_path,
    "cliptype": clipselect.value,
    "visuals": [cb.description for cb in visualselect if cb.value],
    "splittype": splitselect.value if splitselect is not None else None,
    "classtype": classselect.value if classselect is not None else None,
    "classifier": classifierselect.value if classifierselect is not None else None,
    "feature": int(featureselect.value) if featureselect is not None else None,
}

dataset = pd.read_excel(os.path.join(metadata["directory"], "Data", "DatasetComplete.xlsx"), sheet_name='AMD')
dataset.columns = dataset.columns.astype(str).str.strip()

if 'Drusen' in metadata['visuals'] or 'Pigment' in metadata['visuals'] or 'Geographic Atrophy' in metadata['visuals'] or 'Neovascular' in metadata['visuals']:
    fundus = pd.read_excel(os.path.join(metadata["directory"], "Data", "DatasetComplete.xlsx"), sheet_name='fundus')
    fundus.columns = fundus.columns.astype(str).str.strip()

if 'Glaucoma' in metadata['visuals']:
    followup = pd.read_excel(os.path.join(metadata["directory"], "Data", "DatasetComplete.xlsx"), sheet_name='followup')
    followup.columns = followup.columns.astype(str).str.strip()

latents = np.load(os.path.join("results", "areds", "dataset", "encoding", metadata["cliptype"].lower(), "latentvec.npy"))
paths = np.load(os.path.join("results", "areds", "dataset", "encoding", metadata["cliptype"].lower(), "imagepaths.npy"))
paths = pd.DataFrame(paths)
paths = paths.apply(lambda col: col.map(lambda p: os.path.join(metadata["directory"], "Data", "EDO", *Path(p).parts[-5:])))

if metadata["splittype"] is not None and metadata["classtype"]:
    with open(os.path.join("results", "areds", "classifiers", "scalers", metadata["splittype"].split('-')[0].lower().replace(' ', ''), metadata["classtype"].lower().replace(' ', ''), "scaler.pkl"), 'rb') as f: scaler = pkl.load(f)
    latents = scaler.transform(latents)

blacklist = naneval(os.path.join(metadata["directory"], "Data", "EDO"), dataset)
filtpaths = filteringimages(paths, blacklist)
latents = latents[[i for i, path in enumerate(paths[paths.columns[0]].tolist()) if path in filtpaths[filtpaths.columns[0]].tolist()]]
pathlist = filtpaths[filtpaths.columns[0]].tolist()
pointimagepaths = filtpaths.iloc[:, :2].values.tolist()
if 'Set Split' in metadata['visuals']:
    trainpaths = pd.read_csv(os.path.join("results", "areds", "dataset", "splits", f"trainpaths_{metadata['splittype'].split('-')[0].lower().replace(' ', '')}.csv"))
    valpaths = pd.read_csv(os.path.join("results", "areds", "dataset", "splits", f"valpaths_{metadata['splittype'].split('-')[0].lower().replace(' ', '')}.csv"))
    testpaths = pd.read_csv(os.path.join("results", "areds", "dataset", "splits", f"testpaths_{metadata['splittype'].split('-')[0].lower().replace(' ', '')}.csv"))
    setlabels = ['Train' if str(Path(p)) in trainpaths.values else 'Validation' if str(Path(p)) in valpaths.values else 'Test' if str(Path(p)) in testpaths.values else 'NaN' for p in pathlist]
    setlabels = np.array(setlabels)

umap = np.load(os.path.join("results", "areds", "dataset", "encoding", metadata["cliptype"].lower(), "UMAP_NaN.npy"))

coloringtypes = {}
if 'AMD Score' in metadata['visuals']: coloringtypes['AMD Score'] = None
if 'View' in metadata['visuals']: coloringtypes['View'] = None
if 'Classifier' in metadata['visuals']:
    classifierinfo = {"path": os.path.join("results", "areds", "classifiers", "models", metadata["splittype"].split('-')[0].lower().replace(' ', ''), metadata["classtype"].lower().replace(' ', ''), f"{metadata['classifier'].lower().replace(' ', '_')}.pkl")}
    coloringtypes['Classifier'] = classifierinfo
if 'Ground Truth' in metadata['visuals']: coloringtypes['Ground Truth'] = metadata['classtype']
if 'Set Split' in metadata['visuals']: coloringtypes['Set Split'] = setlabels
if 'Drusen' in metadata['visuals']: coloringtypes['Drusen'] = fundus
if 'Pigment' in metadata['visuals']: coloringtypes['Pigment'] = fundus
if 'Geographic Atrophy' in metadata['visuals']: coloringtypes['Geographic Atrophy'] = fundus
if 'Neovascular' in metadata['visuals']: coloringtypes['Neovascular'] = fundus
if 'Glaucoma' in metadata['visuals']: coloringtypes['Glaucoma'] = followup
if 'Diabetes' in metadata['visuals']: coloringtypes['Diabetes'] = followup
if 'Feature' in metadata['visuals']: coloringtypes['Feature'] = int(metadata["feature"])

colorings = {}
for ctype in coloringtypes:
    selected_coloring = coloringextr(ctype, pathlist, dataset, latents, coloringtypes[ctype])
    if selected_coloring is not None: colorings.update(selected_coloring)
fig = umapdisplay(colorings, pathlist, umap, pointimagepaths, figreturn=True)
